### Tools

Models can request to call tools that perform task such as fetching data from the database, searching the web, or runnning code. Tools are pairing of:

1. A Schema, including the name of the tool, description, and/or argument defination(often a JSON format)
2. A function or coroutine to execute

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")

response= model.invoke("Write a story in 50 words")

In [2]:
response

AIMessage(content='As the sun set over the ocean, Emily walked along the beach, collecting seashells. She stumbled upon an old, intricately carved wooden box hidden in the sand. Inside, she found a note from a stranger who had left it there years ago, a message of love and hope that touched her heart.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 42, 'total_tokens': 106, 'completion_time': 0.129246563, 'completion_tokens_details': None, 'prompt_time': 0.00316222, 'prompt_tokens_details': None, 'queue_time': 0.05333725, 'total_time': 0.132408783}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d9f2f-4eeb-7083-b8b6-84050a95a931-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 64, 'total_tokens': 106})

In [3]:
print("Hello")

Hello


In [4]:
## Tools
from langchain.tools import tool

@tool # decorator
def get_weather(location: str) -> str:
    """Get the weather at any location """
    return f"The current weather is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:

response = model_with_tools.invoke("What's the weather in Mumbai?")
print(response)
# response.tool_calls

content='' additional_kwargs={'tool_calls': [{'id': 'bwc4a7aq3', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.023662091, 'completion_tokens_details': None, 'prompt_time': 0.013483923, 'prompt_tokens_details': None, 'queue_time': 0.051831316, 'total_time': 0.037146014}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d9f2f-85a7-7661-a041-398be4727157-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'bwc4a7aq3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}


### Tool execution loop

In [6]:
# step 1: Model generates tools call
messages  = [{"role": "user", "content": "What's the wearher in Mumbai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# messages

# step 2: Execution tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to the LLM model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

However, the function 'get_weather' was used but the exact weather result for Mumbai was not obtained as I am a text-based AI model and do not have the capability to access real-time information.


In [7]:
messages

[{'role': 'user', 'content': "What's the wearher in Mumbai?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9nbzatfsp', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.02740186, 'completion_tokens_details': None, 'prompt_time': 0.015503939, 'prompt_tokens_details': None, 'queue_time': 0.048376185, 'total_time': 0.042905799}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d63f7-5172-7332-8284-52886961a88c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': '9nbzatfsp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}),
 ToolMessage(content='The curr